# 01 - Dataset Overview and City Selection

**Purpose:** choose the city-level scope for the Yelp community-attention forecasting project.

This notebook uses the raw Yelp business file to decide which city is large enough for forecasting, social-network analysis, and NLP feature engineering while keeping execution time reasonable for iteration.

1. Verify that the expected raw Yelp files are available.
2. Count businesses by normalized city/state labels from the business metadata.
3. Save the city-count table used to document scope selection.
4. Select the working city and state.

## Setup

Define project paths and the raw Yelp file names expected by the full pipeline. Only the business file is scanned for city selection, but checking every raw file here catches missing data before the later notebooks do longer streaming passes.


In [1]:
# Configure project paths for local raw data and generated project outputs.
from pathlib import Path
import csv
import json
import re
from collections import Counter, defaultdict

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_YELP_DIR = DATA_DIR / "raw" / "yelp"
OUTPUTS_DIR = PROJECT_ROOT / "outputs"

RAW_FILE_NAMES = [
    "yelp_academic_dataset_business.json",
    "yelp_academic_dataset_review.json",
    "yelp_academic_dataset_user.json",
    "yelp_academic_dataset_checkin.json",
    "yelp_academic_dataset_tip.json",
]

BUSINESS_PATH = RAW_YELP_DIR / RAW_FILE_NAMES[0]
CITY_COUNTS_PATH = OUTPUTS_DIR / "city_business_counts.csv"

OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)

print(PROJECT_ROOT)

c:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp


In [2]:
# Verify file presence before running any longer JSONL scans.
for filename in RAW_FILE_NAMES:
    path = RAW_YELP_DIR / filename
    exists = path.exists()
    exists_label = "yes" if exists else "no"
    size_mb = path.stat().st_size / (1024 * 1024) if exists else 0
    print(f"{filename:<40} exists={exists_label:<3} size_mb={size_mb:>8,.1f}")

yelp_academic_dataset_business.json      exists=yes size_mb=   113.4
yelp_academic_dataset_review.json        exists=yes size_mb= 5,094.4
yelp_academic_dataset_user.json          exists=yes size_mb= 3,207.5
yelp_academic_dataset_checkin.json       exists=yes size_mb=   273.7
yelp_academic_dataset_tip.json           exists=yes size_mb=   172.2


## City Selection Evidence

The output of this section is a deduplicated table with exactly one row per city-state pair:

```text
city,state,business_count
```

To get there, raw Yelp city labels are not used directly as group labels. Each business is first assigned a canonical city key using low-risk cleanup rules: hidden spacing characters, case, punctuation, trailing state abbreviations inside the city field, and common geographic abbreviations such as `St`, `Mt`, `Twp`, `Bch`, `Prt`, `Rchy`, and `Mtng`. The notebook then groups by `(canonical_city_key, state)`, chooses one readable display name for each group, validates that the final `(city, state)` pairs are unique, and saves the table.

This merges labels like `Mount Juliet`/`Mt. Juliet`, `Saint Louis`/`St. Louis`, `O'Fallon`/`O Fallon`, and `Land O Lakes`/`Land O' Lakes`. Cities with the same normalized name in different states remain separate because state is part of the key.


In [3]:
# Build one city-count row per canonical city/state pair.
def iter_jsonl(path):
    with path.open("r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            try:
                yield json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} at line {line_number}") from exc


# Word-level expansions are applied after punctuation is removed.
CITY_WORD_REPLACEMENTS = {
    "mt": "mount",
    "ft": "fort",
    "st": "saint",
    "twp": "township",
    "bch": "beach",
    "prt": "port",
    "pt": "port",
    "rchy": "richey",
    "mtng": "meeting",
    "redngtn": "redington",
    "cntry": "country",
    "twn": "town",
    "hts": "heights",
    "grvs": "groves",
    "sq": "square",
    "terr": "terrace",
    "trev": "trevose",
}

DIRECTION_PREFIXES = {"n": "north", "s": "south", "e": "east", "w": "west"}

def clean_city_text(value):
    return str(value or "").replace("\u00a0", " ").replace("\u200b", "").replace("\ufeff", "")


def display_city(value):
    return " ".join(clean_city_text(value).strip().strip(",").split())


def normalize_state(value):
    return str(value or "").strip().upper()


def normalize_city(value, state=None):
    clean_value = display_city(value).casefold().replace("&", " and ")
    clean_value = re.sub(r"[.,]+", " ", clean_value)
    clean_value = clean_value.replace("\u2019", "'")
    clean_value = re.sub(r"['`/-]+", " ", clean_value)
    clean_value = re.sub(r"\s+", " ", clean_value).strip()

    if state:
        state_key = normalize_state(state).casefold()
        clean_value = re.sub(fr"\b{re.escape(state_key)}\b$", "", clean_value).strip()

    clean_value = re.sub(r"\bofallon\b", "o fallon", clean_value)
    tokens = [CITY_WORD_REPLACEMENTS.get(token, token) for token in clean_value.split()]
    if tokens and tokens[0] in DIRECTION_PREFIXES:
        tokens[0] = DIRECTION_PREFIXES[tokens[0]]
    return " ".join(tokens)


def canonical_city_state(city, state):
    state_key = normalize_state(state)
    city_key = normalize_city(city, state_key)
    city_label = display_city(city)
    return city_key, state_key, city_label


canonical_city_state_counts = Counter()
display_city_counts = defaultdict(Counter)

for record in iter_jsonl(BUSINESS_PATH):
    city_key, state_key, city_label = canonical_city_state(record.get("city"), record.get("state"))
    if not city_key:
        continue

    canonical_city_state_counts[(city_key, state_key)] += 1
    display_city_counts[(city_key, state_key)][city_label] += 1

city_rows = []
for city_state_key, count in canonical_city_state_counts.items():
    _, state_key = city_state_key
    city_label = display_city_counts[city_state_key].most_common(1)[0][0]
    city_rows.append({"city": city_label, "state": state_key, "business_count": count})

city_rows = sorted(city_rows, key=lambda row: (-row["business_count"], row["city"], row["state"]))

output_city_state_keys = [(row["city"], row["state"]) for row in city_rows]
duplicate_output_keys = [
    city_state_key
    for city_state_key, count in Counter(output_city_state_keys).items()
    if count > 1
]
assert not duplicate_output_keys, duplicate_output_keys[:10]


with CITY_COUNTS_PATH.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=["city", "state", "business_count"])
    writer.writeheader()
    writer.writerows(city_rows)
    
print(f"Saved {len(city_rows):,} unique city-state rows: {CITY_COUNTS_PATH}")
print("\nTop 20 canonical city-state rows:")
for row in city_rows[:20]:
    print(f"{row['city']:<24} {row['state']:<4} {row['business_count']:>8}")

Saved 1,164 unique city-state rows: c:\Users\mehdi\OneDrive\Documents\community-forecasting-yelp\outputs\city_business_counts.csv

Top 20 canonical city-state rows:
Philadelphia             PA      14577
Tucson                   AZ       9261
Tampa                    FL       9069
Indianapolis             IN       7546
Nashville                TN       6979
Saint Louis              MO       6461
New Orleans              LA       6215
Reno                     NV       5937
Edmonton                 AB       5056
Santa Barbara            CA       3838
Saint Petersburg         FL       3249
Boise                    ID       2941
Clearwater               FL       2226
Metairie                 LA       1645
Sparks                   NV       1628
Wilmington               DE       1448
Franklin                 TN       1412
Meridian                 ID       1045
Brandon                  FL       1034
Largo                    FL       1007


## Decision

We will use **New Orleans, Louisiana** as city scope. The business count is **6,215**, which is large enough to support business-level time series, social-network exposure features, and review-language features without making the full workflow too heavy.

In [4]:
SELECTED_CITY = "New Orleans"
SELECTED_STATE = "LA"

selected_city_key, selected_state_key, _ = canonical_city_state(SELECTED_CITY, SELECTED_STATE)
selected_city_row = next(
    row
    for row in city_rows
    if normalize_city(row["city"], row["state"]) == selected_city_key and row["state"] == selected_state_key
)


print(f"\nSelected scope: {SELECTED_CITY}, {SELECTED_STATE} ({selected_city_row['business_count']:,} businesses)")


Selected scope: New Orleans, LA (6,215 businesses)
